# NATM Results Analysis

Interactive exploration across all three sectors and all 8 policy/demand
variants: technology diffusion over time, energy demand by carrier over
time (with a segment filter), energy demand by port/hub, the overall fleet
transition summary, financial analysis (capex, NPV, and risk-adjusted
decision metrics from the ambiguity-aware framework), and (when generated)
a real-shape hourly demand profile.

Every chart below carries **Sector** and **Case** dropdowns (Aviation Cargo
/ Maritime Cargo / Aviation Passenger x the 8 policy/demand variants) so you
can switch both interactively without editing code or re-running cells.

For side-by-side comparison across all 8 variants at once (rather than
picking one at a time), see the "Compare all cases" section further down.

In [159]:
from __future__ import annotations

from pathlib import Path

import altair as alt
import pandas as pd

alt.data_transformers.disable_max_rows()

PROJECT_ROOT = Path.cwd().resolve().parents[0] if Path.cwd().name == "analysis" else Path.cwd()
RESULTS_ROOT = PROJECT_ROOT / "data" / "output"

# Maps each sector's case-folder prefix to a human-readable label used
# throughout this notebook (dropdowns, tooltips, chart titles).
SECTOR_CASE_PREFIXES = {
    "Aviation Cargo": "baseline-cargo-transition",
    "Maritime Cargo": "baseline-maritime-cargo-transition",
    "Aviation Passenger": "baseline-passenger-transition",
}

# The 8 policy/demand variants every sector was run for.
SCENARIO_VARIANTS = [
    "aktuellepolitiken-risk-neutral",
    "fokush2-risk-neutral",
    "fokusstrom-risk-neutral",
    "hohenachfrage-risk-neutral",
    "nachfrageniedrig-risk-neutral",
    "technologiemix-risk-neutral",
    "worstcasemean-risk-averse",
    "worstcasetailrisk-risk-averse",
]

found = 0
for sector_group, prefix in SECTOR_CASE_PREFIXES.items():
    for variant in SCENARIO_VARIANTS:
        if (RESULTS_ROOT / f"{prefix}-{variant}").exists():
            found += 1
print(f"Found {found}/{len(SECTOR_CASE_PREFIXES) * len(SCENARIO_VARIANTS)} sector x case combinations.")

import vl_convert as vlc
from IPython.display import display as _display


def show(chart: alt.Chart) -> None:
    """Renders an Altair chart via a multi-mimetype display bundle:

    - application/vnd.vegalite.v5+json: VS Code's and JupyterLab's Jupyter
      renderers draw this natively with their own bundled Vega-Lite --
      no JavaScript execution required, so it works even where a notebook
      frontend sandboxes/strips <script> tags from raw HTML output (the
      "cell runs, no error, but no figure appears" symptom).
    - text/html: a fully self-contained fallback (JS bundled inline via
      vl-convert, no CDN fetch) for frontends without native Vega-Lite
      mimetype support.

    A frontend picks whichever mimetype it understands; unsupported ones
    are ignored, so both can safely be provided together.
    """
    spec = chart.to_dict()
    html = vlc.vegalite_to_html(spec, bundle=True)
    _display(
        {
            "application/vnd.vegalite.v5+json": spec,
            "text/html": html,
            "text/plain": "<Vega-Lite chart>",
        },
        raw=True,
    )


Found 24/24 sector x case combinations.


## Load results

Loads all three sectors x all 8 scenario variants (24 cases) into combined
dataframes, each tagged with `sector_group` and `scenario` columns.
`aircraft_count`/`vessel_count` are unified into a single `stock_count`
column so charts don't need to know which sector they're looking at -- the
Sector and Case dropdowns on each chart just filter `sector_group` and
`scenario`. `aircraft.csv` (per-asset-per-year detail) is large across 24
cases, so only the columns the port/hub chart needs are read.

In [160]:
def _read_csv_if_exists(path: Path, usecols: list[str] | None = None) -> pd.DataFrame:
    if not path.exists():
        return pd.DataFrame()
    if usecols is None:
        return pd.read_csv(path, low_memory=False)
    header = pd.read_csv(path, nrows=0).columns
    return pd.read_csv(path, low_memory=False, usecols=[c for c in usecols if c in header])


AIRCRAFT_USECOLS = [
    "year",
    "main_hub_base",
    "main_hub",
    "primary_energy_carrier",
    "primary_energy_consumption",
    "secondary_energy_consumption",
]

technology_frames: list[pd.DataFrame] = []
energy_frames: list[pd.DataFrame] = []
aircraft_frames: list[pd.DataFrame] = []
investments_frames: list[pd.DataFrame] = []
selected_decisions_frames: list[pd.DataFrame] = []
summary_frames: list[pd.DataFrame] = []

for sector_group, prefix in SECTOR_CASE_PREFIXES.items():
    for variant in SCENARIO_VARIANTS:
        case_dir = RESULTS_ROOT / f"{prefix}-{variant}"
        if not case_dir.exists():
            continue

        this_is_aviation = (case_dir / "aviation_technology.csv").exists()
        this_is_maritime = (case_dir / "maritime_technology.csv").exists()
        if not (this_is_aviation or this_is_maritime):
            continue
        this_sector_prefix = "aviation" if this_is_aviation else "maritime"
        this_stock_col = "aircraft_count" if this_is_aviation else "vessel_count"

        tech = _read_csv_if_exists(case_dir / f"{this_sector_prefix}_technology.csv")
        if not tech.empty:
            tech = tech.rename(columns={this_stock_col: "stock_count"})
            tech["sector_group"] = sector_group
            tech["scenario"] = variant
            technology_frames.append(tech)

        nrg = _read_csv_if_exists(case_dir / f"{this_sector_prefix}_energy_emissions.csv")
        if not nrg.empty:
            nrg = nrg.copy()
            nrg["sector_group"] = sector_group
            nrg["scenario"] = variant
            energy_frames.append(nrg)

        ac = _read_csv_if_exists(case_dir / "aircraft.csv", usecols=AIRCRAFT_USECOLS)
        if not ac.empty:
            ac["sector_group"] = sector_group
            ac["scenario"] = variant
            aircraft_frames.append(ac)

        inv = _read_csv_if_exists(case_dir / f"{this_sector_prefix}_investments.csv")
        if not inv.empty:
            inv = inv.rename(columns={this_stock_col: "stock_count"})
            inv["sector_group"] = sector_group
            inv["scenario"] = variant
            investments_frames.append(inv)

        dec = _read_csv_if_exists(case_dir / "selected_ambiguity_aware_decisions.csv")
        if not dec.empty:
            dec = dec.copy()
            dec["sector_group"] = sector_group
            dec["scenario"] = variant
            selected_decisions_frames.append(dec)

        raw_summary = _read_csv_if_exists(case_dir / "model_summary.csv")
        if not raw_summary.empty:
            s = raw_summary[["year"]].copy()
            s["alternative_share"] = raw_summary[f"{this_sector_prefix}_alternative_share"]
            s["transition_pressure"] = raw_summary[f"{this_sector_prefix}_transition_pressure"]
            s["effective_conventional_cost"] = raw_summary[
                f"{this_sector_prefix}_effective_conventional_cost"
            ]
            s["effective_alternative_cost"] = raw_summary[f"{this_sector_prefix}_effective_alternative_cost"]
            s["sector_group"] = sector_group
            s["scenario"] = variant
            summary_frames.append(s)

technology = pd.concat(technology_frames, ignore_index=True) if technology_frames else pd.DataFrame()
energy = pd.concat(energy_frames, ignore_index=True) if energy_frames else pd.DataFrame()
aircraft = pd.concat(aircraft_frames, ignore_index=True) if aircraft_frames else pd.DataFrame()
investments_by_year = (
    pd.concat(investments_frames, ignore_index=True) if investments_frames else pd.DataFrame()
)
selected_decisions = (
    pd.concat(selected_decisions_frames, ignore_index=True) if selected_decisions_frames else pd.DataFrame()
)
summary = pd.concat(summary_frames, ignore_index=True) if summary_frames else pd.DataFrame()

assert not technology.empty, "No sector/case data found under data/output/"

sector_options = sorted(technology["sector_group"].unique().tolist())
case_options = sorted(technology["scenario"].unique().tolist())
stock_label = "Assets"

print(f"Sectors loaded: {sector_options}")
print(f"Cases loaded: {case_options}")
print(f"Technology rows: {len(technology)}, Energy rows: {len(energy)}, Aircraft rows: {len(aircraft)}")
technology.head()

Sectors loaded: ['Aviation Cargo', 'Aviation Passenger', 'Maritime Cargo']
Cases loaded: ['aktuellepolitiken-risk-neutral', 'fokush2-risk-neutral', 'fokusstrom-risk-neutral', 'hohenachfrage-risk-neutral', 'nachfrageniedrig-risk-neutral', 'technologiemix-risk-neutral', 'worstcasemean-risk-averse', 'worstcasetailrisk-risk-averse']
Technology rows: 28331, Energy rows: 28331, Aircraft rows: 172216


,year,application_name,operator_name,operator_country,segment,current_technology,primary_energy_carrier,secondary_energy_carrier,saf_pathway,stock_count,primary_energy_consumption,secondary_energy_consumption,total_emission,chargeable_emission,sector_group,scenario
0,2025,cargo,AeroLogic,Germany,long,kerosene_freight_long,kerosene,none,none,28,2.679962e+08,0.0,7.064379e+07,7.064379e+07,Aviation Cargo,aktuellepolitiken-risk-neutral
1,2025,cargo,DHL,Germany,long,kerosene_freight_long,kerosene,none,none,2,1.469950e+07,0.0,3.874787e+06,3.874787e+06,Aviation Cargo,aktuellepolitiken-risk-neutral
2,2025,cargo,DHL,Germany,long,kerosene_freight_medium,kerosene,none,none,6,1.180794e+07,0.0,3.112572e+06,3.112572e+06,Aviation Cargo,aktuellepolitiken-risk-neutral
3,2025,cargo,DHL,Germany,medium,kerosene_freight_medium,kerosene,none,none,26,4.964670e+07,0.0,1.308687e+07,1.308687e+07,Aviation Cargo,aktuellepolitiken-risk-neutral
4,2025,cargo,Lufthansa Cargo,Germany,long,kerosene_freight_long,kerosene,none,none,12,5.577696e+07,0.0,1.470281e+07,1.470281e+07,Aviation Cargo,aktuellepolitiken-risk-neutral


## Technology diffusion over time

Stacked area chart of fleet composition by technology, year over year. Use
the **Sector** and **Case** dropdowns to switch between the 3 sectors and 8
policy/demand variants. Click a legend entry to isolate one technology
(shift-click to add more); use the segment dropdown to filter by
short/medium/long or regional/deepsea.

In [161]:
sector_dropdown = alt.binding_select(options=sector_options, name="Sector: ")
sector_select = alt.selection_point(
    fields=["sector_group"],
    bind=sector_dropdown,
    value=[{"sector_group": "Aviation Passenger"}],
)

case_dropdown = alt.binding_select(options=case_options, name="Case: ")
case_select = alt.selection_point(
    fields=["scenario"],
    bind=case_dropdown,
    value=[{"scenario": "aktuellepolitiken-risk-neutral"}],
)

segment_options = ["All"] + sorted(technology["segment"].dropna().astype(str).unique().tolist())
segment_dropdown = alt.binding_select(options=segment_options, name="Segment: ")
segment_select = alt.selection_point(
    fields=["segment_filter"],
    bind=segment_dropdown,
    value=[{"segment_filter": "All"}],
)

technology_plot = technology.copy()
technology_plot["segment_filter"] = technology_plot["segment"]

tech_by_year = (
    technology_plot.groupby(
        ["year", "sector_group", "scenario", "segment_filter", "current_technology"], as_index=False,
    )["stock_count"]
    .sum()
)

# Duplicate rows tagged "All" so the dropdown's "All" option sums every segment.
all_segments = (
    tech_by_year.groupby(
        ["year", "sector_group", "scenario", "current_technology"], as_index=False,
    )["stock_count"]
    .sum()
    .assign(segment_filter="All")
)
tech_by_year = pd.concat([tech_by_year, all_segments], ignore_index=True)

technology_legend_select = alt.selection_point(fields=["current_technology"], bind="legend")

diffusion_chart = (
    alt.Chart(tech_by_year)
    .mark_area()
    .encode(
        x=alt.X("year:O", title="Year"),
        y=alt.Y("stock_count:Q", stack="normalize", title="Share of fleet", axis=alt.Axis(format="%")),
        color=alt.Color("current_technology:N", title="Technology", scale=alt.Scale(scheme="tableau20")),
        opacity=alt.condition(technology_legend_select, alt.value(0.9), alt.value(0.08)),
        tooltip=[
            alt.Tooltip("sector_group:N", title="Sector"),
            alt.Tooltip("scenario:N", title="Case"),
            alt.Tooltip("year:O", title="Year"),
            alt.Tooltip("current_technology:N", title="Technology"),
            alt.Tooltip("stock_count:Q", title=stock_label, format=",.0f"),
        ],
    )
    .add_params(sector_select, case_select, segment_select, technology_legend_select)
    .transform_filter(sector_select)
    .transform_filter(case_select)
    .transform_filter(segment_select)
    .properties(width=760, height=380, title="Technology diffusion over time (share of fleet)")
    .interactive()
)
show(diffusion_chart)

<Vega-Lite chart>

## Technology adoption per year

The diffusion chart above shows fleet *composition* (share of the total
fleet) at each year -- a stock view. This shows the *flow*: how many new
aircraft/vessels were actually invested in with each technology, in each
individual year, from `{sector}_investments.csv`. A technology can dominate
the stock view for years after its adoption flow has already dropped to
zero (fleet turnover is slow), so the two views tell different parts of the
same story. Same Sector and Case dropdowns as above.

In [162]:
if investments_by_year.empty:
    print("No investments CSV found for any sector.")
else:
    adoption_sector_dropdown = alt.binding_select(options=sector_options, name="Sector: ")
    adoption_sector_select = alt.selection_point(
        fields=["sector_group"],
        bind=adoption_sector_dropdown,
        value=[{"sector_group": "Aviation Passenger"}],
    )
    adoption_case_dropdown = alt.binding_select(options=case_options, name="Case: ")
    adoption_case_select = alt.selection_point(
        fields=["scenario"],
        bind=adoption_case_dropdown,
        value=[{"scenario": "aktuellepolitiken-risk-neutral"}],
    )

    adoption_by_year = investments_by_year.groupby(
        ["year", "sector_group", "scenario", "current_technology"], as_index=False,
    )["stock_count"].sum()

    adoption_chart = (
        alt.Chart(adoption_by_year)
        .mark_bar()
        .encode(
            x=alt.X("year:O", title="Year"),
            y=alt.Y("stock_count:Q", title=f"New {stock_label.lower()} adopting technology"),
            color=alt.Color(
                "current_technology:N", title="Technology", scale=alt.Scale(scheme="tableau20"),
            ),
            tooltip=[
                alt.Tooltip("sector_group:N", title="Sector"),
                alt.Tooltip("scenario:N", title="Case"),
                alt.Tooltip("year:O", title="Year"),
                alt.Tooltip("current_technology:N", title="Technology"),
                alt.Tooltip("stock_count:Q", title=stock_label, format=",.0f"),
            ],
        )
        .add_params(adoption_sector_select, adoption_case_select)
        .transform_filter(adoption_sector_select)
        .transform_filter(adoption_case_select)
        .properties(
            width=760,
            height=380,
            title=f"Technology adoption per year (new {stock_label.lower()} by technology)",
        )
    )
    show(adoption_chart)

<Vega-Lite chart>

## Energy demand by carrier over time

Primary + secondary energy consumption per carrier, year over year, in the
model's native `kWh`. Sector and Case dropdowns plus the same segment
filter as above (independent dropdowns rather than a shared cross-chart
selection, since that requires Altair's newer cross-chart param wiring;
this notebook keeps each chart's filters independent so it works reliably
across Altair versions).

In [163]:
energy_plot = energy.copy()
energy_plot["segment_filter"] = energy_plot["segment"]

primary = (
    energy_plot.groupby(
        ["year", "sector_group", "scenario", "segment_filter", "primary_energy_carrier"], as_index=False,
    )["primary_energy_consumption"]
    .sum()
    .rename(columns={"primary_energy_carrier": "carrier", "primary_energy_consumption": "energy_kwh"})
)
secondary_mask = energy_plot["secondary_energy_carrier"].fillna("").astype(str).str.strip().ne("")
secondary_mask &= energy_plot["secondary_energy_carrier"].astype(str).str.lower().ne("none")
secondary = (
    energy_plot.loc[secondary_mask]
    .groupby(
        ["year", "sector_group", "scenario", "segment_filter", "secondary_energy_carrier"], as_index=False,
    )["secondary_energy_consumption"]
    .sum()
    .rename(
        columns={
            "secondary_energy_carrier": "carrier",
            "secondary_energy_consumption": "energy_kwh",
        },
    )
)
energy_by_carrier = pd.concat([primary, secondary], ignore_index=True)
energy_by_carrier = energy_by_carrier.groupby(
    ["year", "sector_group", "scenario", "segment_filter", "carrier"], as_index=False,
)["energy_kwh"].sum()

all_segments_energy = (
    energy_by_carrier.groupby(
        ["year", "sector_group", "scenario", "carrier"], as_index=False,
    )["energy_kwh"]
    .sum()
    .assign(segment_filter="All")
)
energy_by_carrier = pd.concat([energy_by_carrier, all_segments_energy], ignore_index=True)
energy_by_carrier["energy_twh"] = energy_by_carrier["energy_kwh"] / 1_000_000_000.0

energy_sector_dropdown = alt.binding_select(options=sector_options, name="Sector: ")
energy_sector_select = alt.selection_point(
    fields=["sector_group"],
    bind=energy_sector_dropdown,
    value=[{"sector_group": "Aviation Passenger"}],
)
energy_case_dropdown = alt.binding_select(options=case_options, name="Case: ")
energy_case_select = alt.selection_point(
    fields=["scenario"],
    bind=energy_case_dropdown,
    value=[{"scenario": "aktuellepolitiken-risk-neutral"}],
)
energy_segment_dropdown = alt.binding_select(options=segment_options, name="Segment: ")
energy_segment_select = alt.selection_point(
    fields=["segment_filter"],
    bind=energy_segment_dropdown,
    value=[{"segment_filter": "All"}],
)
carrier_legend_select = alt.selection_point(fields=["carrier"], bind="legend")

energy_chart = (
    alt.Chart(energy_by_carrier)
    .mark_area()
    .encode(
        x=alt.X("year:O", title="Year"),
        y=alt.Y("energy_twh:Q", title="Energy demand (TWh)", stack="zero"),
        color=alt.Color("carrier:N", title="Energy carrier", scale=alt.Scale(scheme="set2")),
        opacity=alt.condition(carrier_legend_select, alt.value(0.9), alt.value(0.08)),
        tooltip=[
            alt.Tooltip("sector_group:N", title="Sector"),
            alt.Tooltip("scenario:N", title="Case"),
            alt.Tooltip("year:O", title="Year"),
            alt.Tooltip("carrier:N", title="Carrier"),
            alt.Tooltip("energy_twh:Q", title="Energy (TWh)", format=",.3f"),
        ],
    )
    .add_params(energy_sector_select, energy_case_select, energy_segment_select, carrier_legend_select)
    .transform_filter(energy_sector_select)
    .transform_filter(energy_case_select)
    .transform_filter(energy_segment_select)
    .properties(width=760, height=380, title="Energy demand by carrier over time")
    .interactive()
)
show(energy_chart)

<Vega-Lite chart>

## Energy demand by port / hub

Real per-asset hub/port data (`main_hub_base`, asset-level and
activity-profile-derived for aviation; falls back to `main_hub` -- the home
port for maritime) from `aircraft.csv`, broken down by energy carrier. Use
the Sector and Case dropdowns and the year slider to see how demand shifts
between hubs/ports and carriers over the simulation horizon.

In [164]:
assert not aircraft.empty, "No aircraft.csv found for any sector"

hub_column = "main_hub_base" if "main_hub_base" in aircraft.columns else "main_hub"
port_energy = aircraft.copy()
port_energy["hub"] = port_energy[hub_column].fillna("").astype(str).str.strip()
port_energy = port_energy.loc[port_energy["hub"].ne("")]
port_energy["carrier"] = port_energy["primary_energy_carrier"].fillna("").astype(str).str.lower()
port_energy["energy_kwh"] = pd.to_numeric(
    port_energy.get("primary_energy_consumption", 0.0), errors="coerce",
).fillna(0.0) + pd.to_numeric(
    port_energy.get("secondary_energy_consumption", 0.0), errors="coerce",
).fillna(0.0)

port_energy_by_year = (
    port_energy.groupby(
        ["year", "sector_group", "scenario", "hub", "carrier"], as_index=False,
    )["energy_kwh"]
    .sum()
)
port_energy_by_year["energy_gwh"] = port_energy_by_year["energy_kwh"] / 1_000_000.0

port_sector_dropdown = alt.binding_select(options=sector_options, name="Sector: ")
port_sector_select = alt.selection_point(
    fields=["sector_group"],
    bind=port_sector_dropdown,
    value=[{"sector_group": "Aviation Passenger"}],
)
port_case_dropdown = alt.binding_select(options=case_options, name="Case: ")
port_case_select = alt.selection_point(
    fields=["scenario"],
    bind=port_case_dropdown,
    value=[{"scenario": "aktuellepolitiken-risk-neutral"}],
)

port_years = sorted(port_energy_by_year["year"].dropna().astype(int).unique().tolist())
port_year_select = alt.selection_point(
    fields=["year"],
    bind=alt.binding_range(min=port_years[0], max=port_years[-1], step=1, name="Year: "),
    value=[{"year": port_years[-1]}],
)

port_chart = (
    alt.Chart(port_energy_by_year)
    .mark_bar()
    .encode(
        x=alt.X("energy_gwh:Q", title="Energy demand (GWh)"),
        y=alt.Y("hub:N", title="Hub / port", sort="-x"),
        color=alt.Color("carrier:N", title="Energy carrier", scale=alt.Scale(scheme="set2")),
        tooltip=[
            alt.Tooltip("sector_group:N", title="Sector"),
            alt.Tooltip("scenario:N", title="Case"),
            alt.Tooltip("year:O", title="Year"),
            alt.Tooltip("hub:N", title="Hub / port"),
            alt.Tooltip("carrier:N", title="Carrier"),
            alt.Tooltip("energy_gwh:Q", title="Energy (GWh)", format=",.2f"),
        ],
    )
    .add_params(port_sector_select, port_case_select, port_year_select)
    .transform_filter(port_sector_select)
    .transform_filter(port_case_select)
    .transform_filter(port_year_select)
    .properties(
        width=760,
        height=max(220, 24 * port_energy_by_year["hub"].nunique()),
        title="Energy demand by port / hub (drag the year slider)",
    )
)
show(port_chart)

<Vega-Lite chart>

## Fleet transition summary

Alternative-technology share and transition pressure across the run, for
the sector and case selected in the dropdowns.

In [165]:
transition_sector_dropdown = alt.binding_select(options=sector_options, name="Sector: ")
transition_sector_select = alt.selection_point(
    fields=["sector_group"],
    bind=transition_sector_dropdown,
    value=[{"sector_group": "Aviation Passenger"}],
)
transition_case_dropdown = alt.binding_select(options=case_options, name="Case: ")
transition_case_select = alt.selection_point(
    fields=["scenario"],
    bind=transition_case_dropdown,
    value=[{"scenario": "aktuellepolitiken-risk-neutral"}],
)

transition_long = summary.melt(
    id_vars=["year", "sector_group", "scenario"],
    value_vars=["alternative_share", "transition_pressure"],
    var_name="metric",
    value_name="value",
)
transition_long["metric"] = transition_long["metric"].map(
    {"alternative_share": "Alternative share", "transition_pressure": "Transition pressure"},
)

transition_chart = (
    alt.Chart(transition_long)
    .mark_line(point=True)
    .encode(
        x=alt.X("year:O", title="Year"),
        y=alt.Y("value:Q", title="Share", axis=alt.Axis(format="%")),
        color=alt.Color("metric:N", title="Metric"),
        tooltip=[
            alt.Tooltip("sector_group:N", title="Sector"),
            alt.Tooltip("scenario:N", title="Case"),
            alt.Tooltip("year:O", title="Year"),
            alt.Tooltip("metric:N", title="Metric"),
            alt.Tooltip("value:Q", title="Value", format=".1%"),
        ],
    )
    .add_params(transition_sector_select, transition_case_select)
    .transform_filter(transition_sector_select)
    .transform_filter(transition_case_select)
    .properties(width=760, height=340, title="Alternative share & transition pressure over time")
    .interactive()
)
show(transition_chart)

<Vega-Lite chart>

## Operator-level fleet stock

Final-year fleet composition by operator and technology, for a quick sanity
check of who is flying/sailing what. Sector and Case dropdowns select which
sector/scenario's operators are shown.

In [166]:
operator_sector_dropdown = alt.binding_select(options=sector_options, name="Sector: ")
operator_sector_select = alt.selection_point(
    fields=["sector_group"],
    bind=operator_sector_dropdown,
    value=[{"sector_group": "Aviation Passenger"}],
)
operator_case_dropdown = alt.binding_select(options=case_options, name="Case: ")
operator_case_select = alt.selection_point(
    fields=["scenario"],
    bind=operator_case_dropdown,
    value=[{"scenario": "aktuellepolitiken-risk-neutral"}],
)

final_year = int(technology["year"].max())
final_year_stock = (
    technology.loc[technology["year"] == final_year]
    .groupby(
        ["sector_group", "scenario", "operator_name", "current_technology"], as_index=False,
    )["stock_count"]
    .sum()
)

operator_chart = (
    alt.Chart(final_year_stock)
    .mark_bar()
    .encode(
        x=alt.X("stock_count:Q", title=stock_label),
        y=alt.Y("operator_name:N", title="Operator", sort="-x"),
        color=alt.Color("current_technology:N", title="Technology", scale=alt.Scale(scheme="tableau20")),
        tooltip=[
            alt.Tooltip("sector_group:N", title="Sector"),
            alt.Tooltip("scenario:N", title="Case"),
            alt.Tooltip("operator_name:N", title="Operator"),
            alt.Tooltip("current_technology:N", title="Technology"),
            alt.Tooltip("stock_count:Q", title=stock_label, format=",.0f"),
        ],
    )
    .add_params(operator_sector_select, operator_case_select)
    .transform_filter(operator_sector_select)
    .transform_filter(operator_case_select)
    .properties(
        width=760,
        height=max(220, 28 * final_year_stock["operator_name"].nunique()),
        title=f"Fleet by operator and technology ({final_year})",
    )
)
show(operator_chart)

<Vega-Lite chart>

## Financial analysis

Real financial outputs from the simulation's ambiguity-aware, NPV-based
decision framework: capex spending, the NPV of the technology actually
chosen at each replacement/growth decision, and how that expected NPV
compares against the framework's own worst-case and regret metrics (the
risk-aware side of the same decisions).

### Capex spending over time

Real investment cost (`investment_cost_eur`) by year and technology, from
`{sector}_investments.csv`. Sector and Case dropdowns as above.

In [167]:
if investments_by_year.empty:
    print("No investments CSV found for any sector.")
else:
    capex_sector_dropdown = alt.binding_select(options=sector_options, name="Sector: ")
    capex_sector_select = alt.selection_point(
        fields=["sector_group"],
        bind=capex_sector_dropdown,
        value=[{"sector_group": "Aviation Passenger"}],
    )
    capex_case_dropdown = alt.binding_select(options=case_options, name="Case: ")
    capex_case_select = alt.selection_point(
        fields=["scenario"],
        bind=capex_case_dropdown,
        value=[{"scenario": "aktuellepolitiken-risk-neutral"}],
    )

    capex_by_year = investments_by_year.groupby(
        ["year", "sector_group", "scenario", "current_technology"], as_index=False,
    )["investment_cost_eur"].sum()
    capex_by_year["investment_cost_eur_m"] = capex_by_year["investment_cost_eur"] / 1_000_000.0

    capex_chart = (
        alt.Chart(capex_by_year)
        .mark_bar()
        .encode(
            x=alt.X("year:O", title="Year"),
            y=alt.Y("investment_cost_eur_m:Q", title="Investment (EUR millions)"),
            color=alt.Color("current_technology:N", title="Technology", scale=alt.Scale(scheme="tableau20")),
            tooltip=[
                alt.Tooltip("sector_group:N", title="Sector"),
                alt.Tooltip("scenario:N", title="Case"),
                alt.Tooltip("year:O", title="Year"),
                alt.Tooltip("current_technology:N", title="Technology"),
                alt.Tooltip("investment_cost_eur_m:Q", title="Investment (EUR M)", format=",.1f"),
            ],
        )
        .add_params(capex_sector_select, capex_case_select)
        .transform_filter(capex_sector_select)
        .transform_filter(capex_case_select)
        .properties(width=760, height=380, title="Capex spending over time")
    )
    show(capex_chart)

<Vega-Lite chart>

### NPV of selected decisions over time

`selected_ambiguity_aware_decisions.csv` records, for every replacement or
growth decision, the expected NPV of the technology actually chosen (as
opposed to `ambiguity_decision_scores.csv`, which has every *candidate*
technology considered, selected or not). Plotted here as the mean expected
NPV per year across all decisions made that year, for the sector/case
selected.

In [168]:
if selected_decisions.empty:
    print(
        "No selected_ambiguity_aware_decisions.csv found -- these runs may not use "
        "investment_logic=ambiguity_aware_utility."
    )
else:
    npv_sector_dropdown = alt.binding_select(options=sector_options, name="Sector: ")
    npv_sector_select = alt.selection_point(
        fields=["sector_group"],
        bind=npv_sector_dropdown,
        value=[{"sector_group": "Aviation Passenger"}],
    )
    npv_case_dropdown = alt.binding_select(options=case_options, name="Case: ")
    npv_case_select = alt.selection_point(
        fields=["scenario"],
        bind=npv_case_dropdown,
        value=[{"scenario": "aktuellepolitiken-risk-neutral"}],
    )

    npv_by_year = (
        selected_decisions.groupby(
            ["selected_decision_year", "sector_group", "scenario"], as_index=False,
        )["selected_expected_npv"]
        .mean()
        .rename(columns={"selected_decision_year": "year"})
    )
    npv_by_year["expected_npv_m"] = npv_by_year["selected_expected_npv"] / 1_000_000.0

    npv_chart = (
        alt.Chart(npv_by_year)
        .mark_line(point=True)
        .encode(
            x=alt.X("year:O", title="Year"),
            y=alt.Y("expected_npv_m:Q", title="Mean expected NPV (EUR millions)"),
            tooltip=[
                alt.Tooltip("sector_group:N", title="Sector"),
                alt.Tooltip("scenario:N", title="Case"),
                alt.Tooltip("year:O", title="Year"),
                alt.Tooltip("expected_npv_m:Q", title="Mean expected NPV (EUR M)", format=",.1f"),
            ],
        )
        .add_params(npv_sector_select, npv_case_select)
        .transform_filter(npv_sector_select)
        .transform_filter(npv_case_select)
        .properties(
            width=760,
            height=340,
            title="Mean expected NPV of selected technology decisions over time",
        )
        .interactive()
    )
    show(npv_chart)

<Vega-Lite chart>

### Expected vs. risk-adjusted NPV

The same selected decisions, but comparing the plain expected NPV against
the ambiguity-aware framework's worst-case-mean NPV and max regret --
showing how much of the "expected" return the risk-averse view of the same
decision actually gives up. Sector and Case dropdowns as above.

In [169]:
if selected_decisions.empty:
    print("No selected_ambiguity_aware_decisions.csv found; skipping.")
else:
    risk_sector_dropdown = alt.binding_select(options=sector_options, name="Sector: ")
    risk_sector_select = alt.selection_point(
        fields=["sector_group"],
        bind=risk_sector_dropdown,
        value=[{"sector_group": "Aviation Passenger"}],
    )
    risk_case_dropdown = alt.binding_select(options=case_options, name="Case: ")
    risk_case_select = alt.selection_point(
        fields=["scenario"],
        bind=risk_case_dropdown,
        value=[{"scenario": "aktuellepolitiken-risk-neutral"}],
    )

    risk_columns = {
        "selected_expected_npv": "Expected NPV",
        "selected_robust_worst_case_mean_npv": "Worst-case-mean NPV",
        "selected_max_regret": "Max regret",
    }
    risk_by_year = (
        selected_decisions.groupby(
            ["selected_decision_year", "sector_group", "scenario"], as_index=False,
        )[list(risk_columns)]
        .mean()
        .rename(columns={"selected_decision_year": "year"})
    )
    risk_long = risk_by_year.melt(
        id_vars=["year", "sector_group", "scenario"], var_name="metric", value_name="value_eur",
    )
    risk_long["metric"] = risk_long["metric"].map(risk_columns)
    risk_long["value_eur_m"] = risk_long["value_eur"] / 1_000_000.0

    risk_chart = (
        alt.Chart(risk_long)
        .mark_line(point=True)
        .encode(
            x=alt.X("year:O", title="Year"),
            y=alt.Y("value_eur_m:Q", title="EUR millions"),
            color=alt.Color("metric:N", title="Metric"),
            tooltip=[
                alt.Tooltip("sector_group:N", title="Sector"),
                alt.Tooltip("scenario:N", title="Case"),
                alt.Tooltip("year:O", title="Year"),
                alt.Tooltip("metric:N", title="Metric"),
                alt.Tooltip("value_eur_m:Q", title="EUR M", format=",.1f"),
            ],
        )
        .add_params(risk_sector_select, risk_case_select)
        .transform_filter(risk_sector_select)
        .transform_filter(risk_case_select)
        .properties(
            width=760,
            height=380,
            title="Expected vs. risk-adjusted NPV of selected decisions",
        )
        .interactive()
    )
    show(risk_chart)

<Vega-Lite chart>

### Effective operating cost: conventional vs. alternative

From `model_summary.csv` -- the crossover point where the alternative
technology's effective operating cost drops below the conventional
technology's is the point at which the transition becomes economically
self-sustaining rather than policy/mandate-driven. Sector and Case
dropdowns as above.

In [170]:
cost_sector_dropdown = alt.binding_select(options=sector_options, name="Sector: ")
cost_sector_select = alt.selection_point(
    fields=["sector_group"],
    bind=cost_sector_dropdown,
    value=[{"sector_group": "Aviation Passenger"}],
)
cost_case_dropdown = alt.binding_select(options=case_options, name="Case: ")
cost_case_select = alt.selection_point(
    fields=["scenario"],
    bind=cost_case_dropdown,
    value=[{"scenario": "aktuellepolitiken-risk-neutral"}],
)

cost_long = summary.melt(
    id_vars=["year", "sector_group", "scenario"],
    value_vars=["effective_conventional_cost", "effective_alternative_cost"],
    var_name="technology_path",
    value_name="cost_eur",
)
cost_long["technology_path"] = cost_long["technology_path"].map(
    {"effective_conventional_cost": "Conventional", "effective_alternative_cost": "Alternative"},
)
cost_long["cost_eur_m"] = cost_long["cost_eur"] / 1_000_000.0

cost_chart = (
    alt.Chart(cost_long)
    .mark_line(point=True)
    .encode(
        x=alt.X("year:O", title="Year"),
        y=alt.Y("cost_eur_m:Q", title="Effective operating cost (EUR millions)"),
        color=alt.Color(
            "technology_path:N",
            title="Technology path",
            scale=alt.Scale(domain=["Conventional", "Alternative"], range=["#7f7f7f", "#2a9d8f"]),
        ),
        tooltip=[
            alt.Tooltip("sector_group:N", title="Sector"),
            alt.Tooltip("scenario:N", title="Case"),
            alt.Tooltip("year:O", title="Year"),
            alt.Tooltip("technology_path:N", title="Path"),
            alt.Tooltip("cost_eur_m:Q", title="Cost (EUR M)", format=",.1f"),
        ],
    )
    .add_params(cost_sector_select, cost_case_select)
    .transform_filter(cost_sector_select)
    .transform_filter(cost_case_select)
    .properties(
        width=760,
        height=340,
        title="Effective operating cost: conventional vs. alternative technology",
    )
    .interactive()
)
show(cost_chart)

<Vega-Lite chart>

## Hourly energy timeseries

The full hourly-resolved dataset (per aircraft/vessel, year, energy type,
region) lives in `hourly_energy_timeseries.parquet` inside each case's
results folder, generated separately with:

```bash
natm-hourly-timeseries --results data/output/<run-name>
```

It disaggregates each asset's real annual energy total using a real
hour-of-week activity shape derived from historical OpenSky flight
timestamps (aviation) or Global Fishing Watch AIS port-visit timestamps
(maritime) -- summing any asset's hourly values back up reproduces its
annual total exactly. These files are large (tens of millions of rows per
sector/run), so this notebook only loads small, pre-aggregated slices via
`pyarrow.parquet` filters/columns rather than reading them whole, for
whichever sector x case combinations have one generated.

In [171]:
import pyarrow.parquet as pq

hourly_frames: list[pd.DataFrame] = []
latest_year = int(technology["year"].max())
missing_hourly: list[str] = []

for sector_group, prefix in SECTOR_CASE_PREFIXES.items():
    for variant in SCENARIO_VARIANTS:
        hourly_path = RESULTS_ROOT / f"{prefix}-{variant}" / "hourly_energy_timeseries.parquet"
        if not hourly_path.exists():
            missing_hourly.append(f"{sector_group} / {variant}")
            continue

        pf = pq.ParquetFile(hourly_path)
        hourly_year = pd.read_parquet(
            hourly_path,
            columns=["timestamp", "energy_type", "energy_kwh"],
            filters=[("year", "=", latest_year)],
        )
        hourly_year["sector_group"] = sector_group
        hourly_year["scenario"] = variant
        hourly_frames.append(hourly_year)

if missing_hourly:
    print(f"No hourly timeseries generated for {len(missing_hourly)} sector x case combinations.")
    print("Generate one with: natm-hourly-timeseries --results data/output/<run-name>")

if not hourly_frames:
    print("\nNo hourly timeseries found for any sector/case combination.")
else:
    hourly_all = pd.concat(hourly_frames, ignore_index=True)
    hourly_all["hour_of_day"] = hourly_all["timestamp"].dt.hour
    profile = hourly_all.groupby(
        ["hour_of_day", "sector_group", "scenario", "energy_type"], as_index=False,
    )["energy_kwh"].mean()

    hourly_sector_options = sorted(profile["sector_group"].unique().tolist())
    hourly_case_options = sorted(profile["scenario"].unique().tolist())

    hourly_sector_dropdown = alt.binding_select(options=hourly_sector_options, name="Sector: ")
    hourly_sector_select = alt.selection_point(
        fields=["sector_group"],
        bind=hourly_sector_dropdown,
        value=[{"sector_group": hourly_sector_options[0]}],
    )
    hourly_case_dropdown = alt.binding_select(options=hourly_case_options, name="Case: ")
    hourly_case_select = alt.selection_point(
        fields=["scenario"],
        bind=hourly_case_dropdown,
        value=[{"scenario": hourly_case_options[0]}],
    )

    profile_chart = (
        alt.Chart(profile)
        .mark_line(point=True)
        .encode(
            x=alt.X("hour_of_day:O", title="Hour of day (UTC)"),
            y=alt.Y("energy_kwh:Q", title="Mean hourly energy demand (kWh)"),
            color=alt.Color("energy_type:N", title="Energy type", scale=alt.Scale(scheme="set2")),
            tooltip=[
                alt.Tooltip("sector_group:N", title="Sector"),
                alt.Tooltip("scenario:N", title="Case"),
                alt.Tooltip("hour_of_day:O", title="Hour (UTC)"),
                alt.Tooltip("energy_type:N", title="Energy type"),
                alt.Tooltip("energy_kwh:Q", title="Mean demand (kWh)", format=",.1f"),
            ],
        )
        .add_params(hourly_sector_select, hourly_case_select)
        .transform_filter(hourly_sector_select)
        .transform_filter(hourly_case_select)
        .properties(
            width=760,
            height=340,
            title=f"Real-shape average hourly demand profile ({latest_year})",
        )
    )
    show(profile_chart)

No hourly timeseries generated for 24 sector x case combinations.
Generate one with: natm-hourly-timeseries --results data/output/<run-name>

No hourly timeseries found for any sector/case combination.


# Compare all cases (3 sectors x 8 policy/demand variants)

Every sector -- aviation cargo, maritime cargo, aviation passenger -- was run
for the same 8 policy/demand scenario variants (`aktuellepolitiken`,
`fokush2`, `fokusstrom`, `hohenachfrage`, `nachfrageniedrig`,
`technologiemix`, `worstcasemean-risk-averse`, `worstcasetailrisk-risk-averse`),
24 cases total.

This section auto-discovers all 24 `data/output/baseline-<sector>-transition-<variant>`
folders across all three sectors and plots every chart faceted by sector
(one small-multiple panel per sector, independent scales since absolute
fleet size/energy/cost differ a lot across sectors) and colored by scenario
-- so every case and every sector is visible together in one view, with no
variable to edit and no cell to re-run per sector.

In [172]:
def _scenario_label(prefix: str, path: Path) -> str:
    return path.name[len(prefix) + 1 :]


compare_summaries: list[pd.DataFrame] = []
compare_energy: list[pd.DataFrame] = []
compare_investments: list[pd.DataFrame] = []

for group_label, prefix in SECTOR_CASE_PREFIXES.items():
    case_dirs = sorted(p for p in RESULTS_ROOT.glob(f"{prefix}-*") if p.is_dir())
    if not case_dirs:
        print(f"{group_label}: no scenario variants found for '{prefix}-*'")
        continue
    print(f"{group_label}: {len(case_dirs)} scenario variants")

    for path in case_dirs:
        label = _scenario_label(prefix, path)
        this_is_aviation = (path / "aviation_technology.csv").exists()
        this_is_maritime = (path / "maritime_technology.csv").exists()
        if not (this_is_aviation or this_is_maritime):
            print(f"  Skipping {label}: no aviation_technology.csv or maritime_technology.csv found.")
            continue
        this_sector_prefix = "aviation" if this_is_aviation else "maritime"

        raw_summary = _read_csv_if_exists(path / "model_summary.csv")
        if not raw_summary.empty:
            scenario_summary = raw_summary[["year"]].copy()
            scenario_summary["alternative_share"] = raw_summary[f"{this_sector_prefix}_alternative_share"]
            scenario_summary["transition_pressure"] = raw_summary[
                f"{this_sector_prefix}_transition_pressure"
            ]
            scenario_summary["sector_group"] = group_label
            scenario_summary["scenario"] = label
            compare_summaries.append(scenario_summary)

        scenario_energy = _read_csv_if_exists(path / f"{this_sector_prefix}_energy_emissions.csv")
        if not scenario_energy.empty:
            scenario_energy = scenario_energy.copy()
            scenario_energy["sector_group"] = group_label
            scenario_energy["scenario"] = label
            compare_energy.append(scenario_energy)

        scenario_investments = _read_csv_if_exists(path / f"{this_sector_prefix}_investments.csv")
        if not scenario_investments.empty:
            scenario_investments = scenario_investments.copy()
            scenario_investments["sector_group"] = group_label
            scenario_investments["scenario"] = label
            compare_investments.append(scenario_investments)

compare_summary = pd.concat(compare_summaries, ignore_index=True) if compare_summaries else pd.DataFrame()
compare_energy_df = pd.concat(compare_energy, ignore_index=True) if compare_energy else pd.DataFrame()
compare_investments_df = (
    pd.concat(compare_investments, ignore_index=True) if compare_investments else pd.DataFrame()
)

n_cases = (
    compare_summary[["sector_group", "scenario"]].drop_duplicates().shape[0]
    if not compare_summary.empty
    else 0
)
print(f"\nTotal cases loaded: {n_cases}")
print(f"Combined summary rows: {len(compare_summary)}, energy rows: {len(compare_energy_df)}")

Aviation Cargo: 8 scenario variants
Maritime Cargo: 8 scenario variants
Aviation Passenger: 8 scenario variants

Total cases loaded: 24
Combined summary rows: 624, energy rows: 28331


## Alternative technology share over time, all sectors and scenarios

In [173]:
share_compare_chart = (
    alt.Chart(compare_summary)
    .mark_line(point=True)
    .encode(
        x=alt.X("year:O", title="Year"),
        y=alt.Y("alternative_share:Q", title="Alternative technology share", axis=alt.Axis(format="%")),
        color=alt.Color("scenario:N", title="Scenario", scale=alt.Scale(scheme="tableau20")),
        tooltip=[
            alt.Tooltip("sector_group:N", title="Sector"),
            alt.Tooltip("year:O", title="Year"),
            alt.Tooltip("scenario:N", title="Scenario"),
            alt.Tooltip("alternative_share:Q", title="Alternative share", format=".1%"),
        ],
    )
    .properties(width=230, height=280)
    .facet(
        column=alt.Column("sector_group:N", title=None),
        title="Alternative technology share by scenario, all sectors",
    )
    .resolve_scale(y="independent")
)
show(share_compare_chart)

<Vega-Lite chart>

## Transition pressure over time, all sectors and scenarios

In [174]:
pressure_compare_chart = (
    alt.Chart(compare_summary)
    .mark_line(point=True)
    .encode(
        x=alt.X("year:O", title="Year"),
        y=alt.Y("transition_pressure:Q", title="Transition pressure", axis=alt.Axis(format="%")),
        color=alt.Color("scenario:N", title="Scenario", scale=alt.Scale(scheme="tableau20")),
        tooltip=[
            alt.Tooltip("sector_group:N", title="Sector"),
            alt.Tooltip("year:O", title="Year"),
            alt.Tooltip("scenario:N", title="Scenario"),
            alt.Tooltip("transition_pressure:Q", title="Transition pressure", format=".1%"),
        ],
    )
    .properties(width=230, height=280)
    .facet(
        column=alt.Column("sector_group:N", title=None),
        title="Transition pressure by scenario, all sectors",
    )
    .resolve_scale(y="independent")
)
show(pressure_compare_chart)

<Vega-Lite chart>

## Total energy demand over time, all sectors and scenarios

Sum of primary + secondary energy consumption across the whole fleet, per
year, per scenario, faceted by sector (independent y-scales since absolute
fleet size differs a lot between sectors).

In [175]:
compare_energy_df["energy_kwh_total"] = pd.to_numeric(
    compare_energy_df.get("primary_energy_consumption", 0.0), errors="coerce",
).fillna(0.0) + pd.to_numeric(
    compare_energy_df.get("secondary_energy_consumption", 0.0), errors="coerce",
).fillna(0.0)

energy_totals_by_scenario = compare_energy_df.groupby(
    ["year", "sector_group", "scenario"], as_index=False,
)["energy_kwh_total"].sum()
energy_totals_by_scenario["energy_twh"] = energy_totals_by_scenario["energy_kwh_total"] / 1_000_000_000.0

energy_compare_chart = (
    alt.Chart(energy_totals_by_scenario)
    .mark_line(point=True)
    .encode(
        x=alt.X("year:O", title="Year"),
        y=alt.Y("energy_twh:Q", title="Total energy demand (TWh)"),
        color=alt.Color("scenario:N", title="Scenario", scale=alt.Scale(scheme="tableau20")),
        tooltip=[
            alt.Tooltip("sector_group:N", title="Sector"),
            alt.Tooltip("year:O", title="Year"),
            alt.Tooltip("scenario:N", title="Scenario"),
            alt.Tooltip("energy_twh:Q", title="Energy (TWh)", format=",.3f"),
        ],
    )
    .properties(width=230, height=280)
    .facet(
        column=alt.Column("sector_group:N", title=None),
        title="Total energy demand by scenario, all sectors",
    )
    .resolve_scale(y="independent")
)
show(energy_compare_chart)

<Vega-Lite chart>

## Total emissions over time, all sectors and scenarios

In [176]:
emissions_by_scenario = compare_energy_df.groupby(
    ["year", "sector_group", "scenario"], as_index=False,
)["total_emission"].sum()
emissions_by_scenario["total_emission_kt"] = emissions_by_scenario["total_emission"] / 1_000_000.0

emissions_compare_chart = (
    alt.Chart(emissions_by_scenario)
    .mark_line(point=True)
    .encode(
        x=alt.X("year:O", title="Year"),
        y=alt.Y("total_emission_kt:Q", title="Total emissions (kt CO2)"),
        color=alt.Color("scenario:N", title="Scenario", scale=alt.Scale(scheme="tableau20")),
        tooltip=[
            alt.Tooltip("sector_group:N", title="Sector"),
            alt.Tooltip("year:O", title="Year"),
            alt.Tooltip("scenario:N", title="Scenario"),
            alt.Tooltip("total_emission_kt:Q", title="Emissions (kt CO2)", format=",.1f"),
        ],
    )
    .properties(width=230, height=280)
    .facet(
        column=alt.Column("sector_group:N", title=None),
        title="Total emissions by scenario, all sectors",
    )
    .resolve_scale(y="independent")
)
show(emissions_compare_chart)

<Vega-Lite chart>

## Capex spending over time, all sectors and scenarios

In [177]:
if compare_investments_df.empty:
    print("No investments CSV found for any scenario variant.")
else:
    capex_by_scenario = compare_investments_df.groupby(
        ["year", "sector_group", "scenario"], as_index=False,
    )["investment_cost_eur"].sum()
    capex_by_scenario["investment_cost_eur_m"] = capex_by_scenario["investment_cost_eur"] / 1_000_000.0

    capex_compare_chart = (
        alt.Chart(capex_by_scenario)
        .mark_line(point=True)
        .encode(
            x=alt.X("year:O", title="Year"),
            y=alt.Y("investment_cost_eur_m:Q", title="Investment (EUR millions)"),
            color=alt.Color("scenario:N", title="Scenario", scale=alt.Scale(scheme="tableau20")),
            tooltip=[
                alt.Tooltip("sector_group:N", title="Sector"),
                alt.Tooltip("year:O", title="Year"),
                alt.Tooltip("scenario:N", title="Scenario"),
                alt.Tooltip("investment_cost_eur_m:Q", title="Investment (EUR M)", format=",.1f"),
            ],
        )
        .properties(width=230, height=280)
        .facet(
            column=alt.Column("sector_group:N", title=None),
            title="Capex spending by scenario, all sectors",
        )
        .resolve_scale(y="independent")
    )
    show(capex_compare_chart)

<Vega-Lite chart>

## Final-year snapshot across all cases

A quick at-a-glance bar comparing the final simulated year's alternative
share directly across every scenario and sector, sorted from lowest to
highest within each sector panel.

In [178]:
final_year_compare = int(compare_summary["year"].max())
final_snapshot = compare_summary.loc[
    compare_summary["year"] == final_year_compare,
    ["sector_group", "scenario", "alternative_share"],
]

snapshot_chart = (
    alt.Chart(final_snapshot)
    .mark_bar()
    .encode(
        x=alt.X("alternative_share:Q", title="Alternative technology share", axis=alt.Axis(format="%")),
        y=alt.Y("scenario:N", title="Scenario", sort="-x"),
        color=alt.Color("scenario:N", title="Scenario", scale=alt.Scale(scheme="tableau20"), legend=None),
        tooltip=[
            alt.Tooltip("sector_group:N", title="Sector"),
            alt.Tooltip("scenario:N", title="Scenario"),
            alt.Tooltip("alternative_share:Q", title="Alternative share", format=".1%"),
        ],
    )
    .properties(width=220, height=max(180, 22 * final_snapshot["scenario"].nunique()))
    .facet(
        column=alt.Column("sector_group:N", title=None),
        title=f"Final-year ({final_year_compare}) alternative share by scenario, all sectors",
    )
    .resolve_scale(x="independent")
)
show(snapshot_chart)

<Vega-Lite chart>